Concluída a etapa de transformação, com a consolidação dos resultados no nível Silver, passamos à etapa referente ao **nível Ouro** do pipeline de dados.

 Para resposta ao questionamento inicial, faz-se necessário quantificar o comportamento da quantidade de processos de execução fiscal ajuizados ao longo do período de apuração deste trabalho, bem como a sua distribuição entre as unidades do Tribunal de Justiça do Estado de São Paulo.

Logo, a entidade central do modelo de dados seriam os processos, os quais têm, entre seus atributos, a data de ajuizamento, que corresponde à data em que ocorreu o seu protocolo. Por outro lado, essa entidade possui relacionamento com a entidade "Órgão Julgador", que representa a unidade judiciária competente para o processamento daquele processo.

A partir desse arranjo, teremos uma **modelagem em estrela** com os seguintes componentes:

1) Tabela-Fato: **Processos**
2) Tabela-Dimensão: **Tempo**
3) Tabela-Dimensão: **ÓrgãoJulgador**

In [0]:
%sql
CREATE OR REPLACE TABLE Gold_OrgaoJulgador AS

SELECT 
    ROW_NUMBER() OVER (ORDER BY orgaoJulgador_nome) AS id_orgao,
    orgaojulgador_nome as nome
FROM (
    SELECT DISTINCT orgaoJulgador_nome
    FROM Silver_processos
);


SELECT COUNT (DISTINCT id_orgao)
FROM Gold_OrgaoJulgador





In [0]:
%sql
CREATE OR REPLACE TABLE Gold_Tempo AS

SELECT DISTINCT
CAST(DATE_FORMAT(TO_DATE(data_ajuizamento), 'yyyyMMdd') AS INT) AS id_data,
ano_mes_ajuizamento,
ano,
mes,
TO_DATE(data_ajuizamento) AS data_ajuizamento
FROM Silver_processos;


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT id_data) AS ids_distintos,
    COUNT(DISTINCT data_ajuizamento) AS datas_distintas,
    MIN(data_ajuizamento) AS menor_data,
    MAX(data_ajuizamento) AS maior_data
FROM Gold_Tempo;

In [0]:
%sql
CREATE OR REPLACE TABLE Gold_FatoAjuizamento AS

SELECT
    p.numero_processo,
    t.id_data,
    o.id_orgao,
    p.grau
FROM Silver_processos p

LEFT JOIN Gold_Tempo t ON p.data_ajuizamento = t.data_ajuizamento

LEFT JOIN Gold_OrgaoJulgador o ON p.orgaoJulgador_nome = o.nome;



In [0]:
%sql
SELECT COUNT (*)
FROM Gold_FatoAjuizamento;